In [1]:
from dotenv import load_dotenv
load_dotenv()

True

In [38]:
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_openai import OpenAIEmbeddings
from langchain_community.vectorstores import InMemoryVectorStore
from langchain_groq import ChatGroq

from langgraph.graph import StateGraph,START,END
from pydantic import BaseModel,Field

In [39]:
loader=PyPDFLoader(file_path= r"D:\bloodtestreport.pdf")
docs=loader.load()

In [40]:
splitter=RecursiveCharacterTextSplitter(chunk_size=500,chunk_overlap=50)
splitted_docs=splitter.split_documents(docs)

In [41]:
embed=OpenAIEmbeddings(model="text-embedding-3-large")

vectorStore=InMemoryVectorStore.from_documents(
    embedding=embed,
    documents=splitted_docs
)

In [42]:
llm=ChatGroq(model="openai/gpt-oss-20b")

In [44]:
class RagState(BaseModel):
    question:str = Field(description="User Question")
    documents:list=[]
    context:str=Field(description="Context data for user question",default="")
    answer:str=Field(description="Final answer...",default="")

#### question -> retrieve -> context -> generate -> end

In [46]:
def retrieve_node(state:RagState)-> RagState:
    docs=vectorStore.similarity_search(query=state.question)
    state.documents=docs
    return state

def context_node(state:RagState)->RagState:
    context = ""
    for doc in state.documents:
        context += doc.page_content + "\n\n"
    state.context = context
    return state

def generate_node(state:RagState)->RagState:
    prompt=f"""
        You are a assistant and provide the answer for user question based on the provided context. 
        If you don't find the answer then just say 'I don't know.
        context is: {state.context}
        question is: {state.question}
    '"""
    res=llm.invoke(prompt)
    state.answer=res.content
    return state

In [47]:
graph=StateGraph(RagState)

graph.add_node("retrieve",retrieve_node)
graph.add_node("context",context_node)
graph.add_node("generate",generate_node)

graph.add_edge(START,"retrieve")
graph.add_edge("retrieve","context")
graph.add_edge("context","generate")
graph.add_edge("generate",END)

graph=graph.compile()

In [59]:
res=graph.invoke({"question":"what is value of rbc and is it in range?"})
print(res["answer"])

The reported Red Blood Cell (RBC) count is **5.00 × 10⁶ / µL**.  
For a 38‑year‑old male, the normal reference range is **4.5–5.5 × 10⁶ / µL**.  

**Result:** The RBC count is within the normal range.
